In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType, StringType

spark = (
    SparkSession.builder
    .appName("binance-etl")
    .master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")     # crypto trades in UTC
    .config("spark.driver.memory", "4g")             # avoid OOM on the write
    .config("spark.sql.shuffle.partitions", "50")
    .getOrCreate()
)

In [ ]:
schema = StructType([
    StructField("open_time",       LongType(),   True),
    StructField("open",            StringType(), True),
    StructField("high",            StringType(), True),
    StructField("low",             StringType(), True),
    StructField("close",           StringType(), True),
    StructField("volume",          StringType(), True),
    StructField("close_time",      LongType(),   True),
    StructField("quote_volume",    StringType(), True),
    StructField("num_trades",      LongType(),   True),
    StructField("taker_buy_base",  StringType(), True),
    StructField("taker_buy_quote", StringType(), True),
    StructField("ignore",          StringType(), True),
])

In [ ]:
files_path = "../data/klines/*/*.csv"

# read all csvs into one df
df = spark.read.csv(files_path, schema=schema, header=False)

# extract Symbol from directories
symbol = F.regexp_extract(F.input_file_name(), r"/klines/([A-Z0-9]+)/", 1)

# create new column Symbol and add symbol to it
df = df.withColumn("symbol", symbol)


print("Total rows:", df.count())

df.sample(fraction=0.0001).show()

In [ ]:
# microseconds -> milliseconds -> real timestamp
df = df.withColumn(
    "open_time_ms",
    F.when(F.col("open_time") > 100_000_000_000_000, F.col("open_time") / 1000)
    .otherwise(F.col("open_time"))
    .cast("long")
)

df = df.withColumn(
    "event_time",
    (F.col("open_time_ms") / 1000)
    .cast("timestamp")
)

# numeric casts for the columns we compute on
df = df.withColumn("close_num", F.col("close").cast("double"))
df = df.withColumn("high_num", F.col("high").cast("double"))
df = df.withColumn("low_num", F.col("low").cast("double"))
df = df.withColumn("volume_num", F.col("volume").cast("double"))
df = df.withColumn("taker_buy_base_num", F.col("taker_buy_base").cast("double"))

df.sample(fraction=0.0001).show()

## Log-returns

A log-return is how much the price moved in one minute: `ln(close_now / close_before)`.

We use logs because they add up over time.

 the next feature `Realised variance` relies on that.


In [ ]:
by_time = Window.partitionBy("symbol").orderBy("event_time")

df = df.withColumn("prev_close_num", F.lag("close_num").over(by_time))

df = df.withColumn("log_return", F.log( 
    F.try_divide(
        F.col("close_num"), 
        F.col("prev_close_num")))
)

df = df.withColumn("sq_return", F.col("log_return") * F.col("log_return"))

In [ ]:
df.sample(fraction=0.0001).show()

## Realised variance over 4 windows

`Realised variance` = sum of `squared log-returns` over a time window.

It measures "how much did the price jostle around," ignoring direction. A calm hour = small value; a news spike = large value.

`rowsBetween(-(n-1), 0)` means "this row plus the n-1 rows before it".

In [ ]:
def rolling_sum(col, minutes):
  window = Window.partitionBy("symbol").orderBy("event_time").rowsBetween(-(minutes - 1), 0)
  return F.sum(col).over(window)

df = df.withColumn("rv_15m",  rolling_sum("sq_return", 15))
df = df.withColumn("rv_1h",   rolling_sum("sq_return", 60))
df = df.withColumn("rv_4h",   rolling_sum("sq_return", 240))
df = df.withColumn("rv_24h",  rolling_sum("sq_return", 1440))

In [ ]:
df.show()

## Parkinson estimator + volume and buy-ratio features

`Parkinson` uses each candle's high-low range to catch volatility that close-to-close returns miss (a bar that swung wildly but closed flat).

`Volume z-score` = how unusual this minute's volume is vs the last hour.

 `Buy ratio` = how much of the volume was aggressive buying.

 Both feed the anomaly detector later.

In [ ]:
def rolling_avg(col, minutes):
  window = Window.partitionBy("symbol").orderBy("event_time").rowsBetween(-(minutes - 1), 0)
  return F.avg(col).over(window)

# Parkinson: per-minute, then averaged over a window
factor = 1.0 / (4.0 * F.log( F.lit(2.0) ) )
df = df.withColumn("parkinson_1m", factor * F.pow(F.log(F.col("high_num") / F.col("low_num")), 2))

df = df.withColumn("parkinson_15m", rolling_avg("parkinson_1m", 15))
df = df.withColumn("parkinson_1h",  rolling_avg("parkinson_1m", 60))

# volume z-score over the trailing hour
vol_win = Window.partitionBy("symbol").orderBy("event_time").rowsBetween(-59, 0)
df = df.withColumn("vol_mean_1h", F.avg("volume_num").over(vol_win))
df = df.withColumn("vol_std_1h",  F.stddev("volume_num").over(vol_win))
df = df.withColumn("volume_zscore", F.try_divide(
    (F.col("volume_num") - F.col("vol_mean_1h")), F.col("vol_std_1h"))
)

# buy ratio = taker buy volume / total volume
df = df.withColumn("buy_ratio", F.try_divide(
    F.col("taker_buy_base_num"), F.col("volume_num"))
)

In [ ]:
df.groupBy("symbol").count().orderBy("symbol").show(25, truncate=False)

total = df.count()
distinct = df.select("symbol", "event_time").distinct().count()
print(f"duplicates: {total - distinct}")

# gaps: more than 60s between consecutive rows
gap_win = Window.partitionBy("symbol").orderBy("event_time")
gaps = (
    df.withColumn("prev_t", F.lag("event_time").over(gap_win))
      .withColumn("gap_s", F.col("event_time").cast("long") - F.col("prev_t").cast("long"))
      .filter(F.col("gap_s") > 60)
)
print("rows with a gap before them:", gaps.count(), "(0 = continuous)")

In [ ]:
(
    df.write
    .mode("overwrite")
    .partitionBy("symbol")
    .parquet("../data/parquet/features")
)
print("saved to data/parquet/features")

In [ ]:
check = spark.read.parquet("../data/parquet/features")
print("rows read back:", check.count())
check.select("symbol", "event_time", "close_num", "rv_1h", "volume_zscore", "buy_ratio").show(5, truncate=False)